# Wardrobe Combinations — OpenAI Image Generator

Generates photorealistic menswear images for all 154 wardrobe combinations using **OpenAI image APIs**.

**No GPU required** — runs on any Colab runtime.

## Setup
1. Add your API key in Colab: **Secrets** (key icon in left sidebar) → add `OPENAI_API_KEY`
2. Upload `wardrobe_mongo.json` when prompted
3. Run the test cell to compare 3 OpenAI models, pick the best, then batch-generate

In [ ]:
# Cell 1: Install dependencies
!pip install -q openai Pillow tqdm

In [ ]:
# Cell 2: API key
import os
from getpass import getpass

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print('Loaded key from Colab Secrets')
except Exception:
    OPENAI_API_KEY = getpass('OpenAI API Key: ')
    print('Loaded key from manual input')

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print('Key set.')

In [ ]:
# Cell 3: Upload wardrobe_mongo.json
import json
from google.colab import files

uploaded = files.upload()

with open('wardrobe_mongo.json', 'r') as f:
    combinations = json.load(f)

print(f'Loaded {len(combinations)} combinations')
print(f'Categories: {set(c["input_json"]["category"] for c in combinations)}')
print(f'Sample: {combinations[0]["combo_id"]}')

In [ ]:
# Cell 4: Prompt template

PROMPT_TEMPLATE = """Photorealistic premium commercial menswear photograph of an adult male model wearing a sophisticated business formal outfit.

MODEL & FRAMING
Adult male model with a natural, realistic physique. Face intentionally cropped out; do not show the full face. Frame the model from approximately the lower neck and chin area down to the shoes. Full outfit must remain visible. Upright, relaxed posture. Both hands placed naturally inside the trouser pockets. Legs positioned in a relaxed fashion-model stance: one leg crossed naturally in front of the other at the ankles. Natural human proportions and anatomy.

SHIRT
{shirt_color} ({shirt_hex}) long-sleeve dress shirt in fine cotton with subtle woven texture. Modern tailored fit. Semi-spread collar. Sleeves rolled neatly to just below the elbows. Top two buttons open, remaining buttons closed. Natural fabric texture and realistic folds. Preserve the specified color accurately under the lighting.

TROUSERS
{trouser_color} ({trouser_hex}) tailored formal trousers in smooth lightweight wool-blend suiting fabric. Modern tailored fit. Medium-high rise. Clean flat front. Natural tailoring and realistic fabric tension.

BELT
{belt_color} ({belt_hex}) simple leather dress belt with a simple rectangular brushed-metal buckle.

SHOES
{shoe_color} ({shoe_hex}) classic {shoe_type} dress shoes in full-grain polished leather with subtle natural reflections. Realistic leather texture.

WATCH
Stainless-steel dress wristwatch with a {dial_color} ({dial_hex}) dial and a {strap_color} ({strap_hex}) leather strap.

POSE
Relaxed, confident business-fashion posture. Both hands inside the trouser pockets. One leg crossed naturally in front of the other at the ankles. Natural weight distribution.

ENVIRONMENT
Upscale modern office interior. Contemporary understated architecture. Warm beige and taupe tones. Softly blurred background with realistic depth.

LIGHTING
Soft diffused indoor lighting. Warm neutral color temperature. Natural-looking shadows. Realistic highlights on leather, metal, and fabric.

CAMERA & PHOTOGRAPHY
Premium commercial menswear photography. DSLR photographic realism. Lens approximately 85mm. Straight-on perspective, slightly above waist height. Moderate shallow depth of field. Subject sharply focused. Background softly blurred. Vertical fashion editorial composition.

DO NOT include: blazer, jacket, tie, pocket square, jeans, sneakers, shorts, oversized clothing, exaggerated muscles, distorted anatomy, extra fingers, deformed hands, floating garments, unrealistic fabric folds, artificial-looking skin, excessive HDR, extreme contrast, dramatic fashion pose, visible full face, logos, brand names, watermark, text, cartoon, anime, illustration, painting, drawing."""


def assemble_prompt(input_json):
    return PROMPT_TEMPLATE.format(**input_json)


sample = combinations[0]['input_json']
prompt = assemble_prompt(sample)
print(f'Prompt length: {len(prompt)} chars, ~{len(prompt.split())} words')

In [ ]:
# Cell 5: Initialize OpenAI client
from openai import OpenAI

client = OpenAI()
print('OpenAI client initialized.')

In [ ]:
# Cell 6: Generate 3 test images for L-001
import os
import base64
import time

os.makedirs('/content/test_output', exist_ok=True)

test_combo = combinations[0]
prompt = assemble_prompt(test_combo['input_json'])
combo_id = test_combo['combo_id']

results = {}

# --- Test 1: gpt-image-2 LOW ---
print('1/3: gpt-image-2 (low)...')
try:
    resp = client.images.generate(
        model='gpt-image-2',
        prompt=prompt,
        size='1024x1536',
        quality='low',
        n=1,
    )
    img_bytes = base64.b64decode(resp.data[0].b64_json)
    path = f'/content/test_output/{combo_id}_img2_low.png'
    with open(path, 'wb') as f:
        f.write(img_bytes)
    results['gpt-image-2 LOW\n~$0.005/img'] = path
    print(f'  Saved: {path}')
except Exception as e:
    print(f'  FAILED: {e}')

time.sleep(2)

# --- Test 2: gpt-image-2 MEDIUM ---
print('2/3: gpt-image-2 (medium)...')
try:
    resp = client.images.generate(
        model='gpt-image-2',
        prompt=prompt,
        size='1024x1536',
        quality='medium',
        n=1,
    )
    img_bytes = base64.b64decode(resp.data[0].b64_json)
    path = f'/content/test_output/{combo_id}_img2_medium.png'
    with open(path, 'wb') as f:
        f.write(img_bytes)
    results['gpt-image-2 MEDIUM\n~$0.041/img'] = path
    print(f'  Saved: {path}')
except Exception as e:
    print(f'  FAILED: {e}')

time.sleep(2)

# --- Test 3: gpt-image-2.5-flare ---
print('3/3: gpt-image-2.5-flare...')
try:
    resp = client.images.generate(
        model='gpt-image-2.5-flare',
        prompt=prompt,
        size='1024x1536',
        quality='medium',
        n=1,
    )
    img_bytes = base64.b64decode(resp.data[0].b64_json)
    path = f'/content/test_output/{combo_id}_img25_flare.png'
    with open(path, 'wb') as f:
        f.write(img_bytes)
    results['gpt-image-2.5-flare\n~$0.037/img'] = path
    print(f'  Saved: {path}')
except Exception as e:
    print(f'  FAILED: {e}')

print(f'\nGenerated {len(results)}/3 test images')

In [ ]:
# Cell 7: Display test images side by side
from IPython.display import display, HTML
from PIL import Image
from io import BytesIO
import base64

html = '<div style="display:flex; flex-wrap:wrap; gap:16px; align-items:flex-start;">'
for label, path in results.items():
    img = Image.open(path)
    img_thumb = img.copy()
    img_thumb.thumbnail((400, 600))
    buf = BytesIO()
    img_thumb.save(buf, format='PNG')
    b64 = base64.b64encode(buf.getvalue()).decode()
    label_html = label.replace('\n', '<br>')
    html += f'''
    <div style="text-align:center;">
      <img src="data:image/png;base64,{b64}" style="border:1px solid #ccc; border-radius:8px;">
      <p style="font-weight:bold; margin-top:8px;">{label_html}</p>
      <p style="color:#666; font-size:12px;">{img.size[0]}x{img.size[1]}</p>
    </div>'''
html += '</div>'
display(HTML(html))

---
## Batch Generation

After comparing the test images above, set your chosen model and quality in the next cell, then run the batch.

In [ ]:
# Cell 8: Configure your chosen model
#
# Uncomment ONE block below based on the test results.

# --- Option A: gpt-image-2 (low) --- ~$0.77 total
# MODEL = 'gpt-image-2'
# QUALITY = 'low'

# --- Option B: gpt-image-2 (medium) --- ~$6.31 total
MODEL = 'gpt-image-2'
QUALITY = 'medium'

# --- Option C: gpt-image-2.5-flare --- ~$5.70 total
# MODEL = 'gpt-image-2.5-flare'
# QUALITY = 'medium'

SIZE = '1024x1536'
print(f'Model: {MODEL}, Quality: {QUALITY}, Size: {SIZE}')

In [ ]:
# Cell 9: Batch generate — Locked → Potential → Doubtful, starting from L-040
# Tracks progress in /content/tracker.csv. Graceful exit on credit limit errors.
import os
import csv
import base64
import time
from tqdm.notebook import tqdm
from openai import RateLimitError

os.makedirs('/content/output', exist_ok=True)

TRACKER_PATH = '/content/tracker.csv'
CATEGORY_ORDER = ['Locked', 'Potential', 'Doubtful']

# Sort combinations by category priority
sorted_combos = []
for cat in CATEGORY_ORDER:
    sorted_combos.extend([c for c in combinations if c['input_json']['category'] == cat])

# Initialize tracker CSV — load existing or create new
completed_ids = set()
if os.path.exists(TRACKER_PATH):
    with open(TRACKER_PATH, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row['status'] == 'done':
                completed_ids.add(row['combo_id'])
    print(f'Loaded tracker: {len(completed_ids)} already completed')
else:
    with open(TRACKER_PATH, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['combo_id', 'category', 'status', 'timestamp'])
        # Pre-mark L-001 through L-039 as done
        for c in sorted_combos:
            cid = c['combo_id']
            cat = c['input_json']['category']
            if cid.startswith('L-') and int(cid.split('-')[1]) <= 39:
                writer.writerow([cid, cat, 'done', '2026-10-04T00:00:00'])
                completed_ids.add(cid)
            else:
                writer.writerow([cid, cat, 'pending', ''])
    print(f'Created tracker: {len(completed_ids)} pre-marked as done (L-001 to L-039)')

# Build the pending queue
pending = [c for c in sorted_combos if c['combo_id'] not in completed_ids]
print(f'Pending: {len(pending)} images to generate\n')

generated = []
skipped = []
failed = []
credit_exhausted = False

for combo in tqdm(pending, desc='Generating images'):
    combo_id = combo['combo_id']
    category = combo['input_json']['category']
    path = f'/content/output/{combo_id}.png'

    if os.path.exists(path):
        skipped.append(combo_id)
        # Update tracker
        completed_ids.add(combo_id)
        continue

    prompt = assemble_prompt(combo['input_json'])

    try:
        resp = client.images.generate(
            model=MODEL,
            prompt=prompt,
            size=SIZE,
            quality=QUALITY,
            n=1,
        )
        img_bytes = base64.b64decode(resp.data[0].b64_json)
        with open(path, 'wb') as f:
            f.write(img_bytes)
        generated.append(combo_id)
        completed_ids.add(combo_id)
        time.sleep(1)

    except RateLimitError as e:
        err_msg = str(e).lower()
        if 'insufficient' in err_msg or 'quota' in err_msg or 'billing' in err_msg:
            print(f'\n*** CREDIT LIMIT REACHED at {combo_id} ({category}) ***')
            print(f'    {e}')
            failed.append((combo_id, 'credit_limit'))
            credit_exhausted = True
            break
        else:
            print(f'Rate limited on {combo_id}, waiting 60s...')
            time.sleep(60)
            failed.append((combo_id, str(e)))

    except Exception as e:
        failed.append((combo_id, str(e)))
        print(f'FAILED {combo_id}: {e}')
        time.sleep(3)

# Rewrite tracker CSV with current state
with open(TRACKER_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['combo_id', 'category', 'status', 'timestamp'])
    now = time.strftime('%Y-%m-%dT%H:%M:%S')
    for c in sorted_combos:
        cid = c['combo_id']
        cat = c['input_json']['category']
        if cid in completed_ids:
            writer.writerow([cid, cat, 'done', now])
        else:
            writer.writerow([cid, cat, 'pending', ''])

# Summary
print(f'\n{"="*50}')
print(f'Generated this run: {len(generated)}')
print(f'Skipped (already existed): {len(skipped)}')
print(f'Failed: {len(failed)}')
print(f'Total completed: {len(completed_ids)} / {len(sorted_combos)}')
print(f'Remaining: {len(sorted_combos) - len(completed_ids)}')
if credit_exhausted:
    print('\n*** Stopped early due to credit limit. Top up credits and re-run this cell to resume. ***')
print(f'\nTracker saved: {TRACKER_PATH}')

In [ ]:
# Cell 10: Download all images as zip
import zipfile
import os
from google.colab import files

zip_path = '/content/wardrobe_images.zip'
output_dir = '/content/output'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(output_dir)):
        if fname.endswith('.png'):
            zf.write(os.path.join(output_dir, fname), fname)

count = len([f for f in os.listdir(output_dir) if f.endswith('.png')])
print(f'Zipped {count} images')
files.download(zip_path)